# Lesson 18: Neural Networks from Scratch

*ML Academy — companion notebook*

Neurons, layers, activations and backpropagation — built by hand in NumPy.

Run each cell in order with **Shift+Enter**. The explanations, maths and worked examples are in `lesson-18.html`; this notebook contains every code example from the lesson so you can run and modify it.

Requirements: `pip install numpy pandas scikit-learn matplotlib`

## 11. In Python: a network from scratch

In [1]:
import numpy as np

X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
Y = np.array([[0], [1], [1], [0]], dtype=float)

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

rng = np.random.default_rng(1)
W1 = rng.normal(0, 1, (2, 4)); b1 = np.zeros(4)     # input -> 4 hidden neurons
W2 = rng.normal(0, 1, (4, 1)); b2 = np.zeros(1)     # hidden -> output
lr = 0.5

for step in range(5001):
    # forward pass
    H = np.tanh(X @ W1 + b1)
    P = sigmoid(H @ W2 + b2)
    loss = -np.mean(Y * np.log(P) + (1 - Y) * np.log(1 - P))
    # backward pass (backpropagation)
    dZ2 = (P - Y) / len(X)
    dW2 = H.T @ dZ2;  db2 = dZ2.sum(axis=0)
    dZ1 = (dZ2 @ W2.T) * (1 - H ** 2)          # tanh'(z) = 1 - tanh(z)^2
    dW1 = X.T @ dZ1;  db1 = dZ1.sum(axis=0)
    # gradient descent update
    W1 -= lr * dW1;  b1 -= lr * db1
    W2 -= lr * dW2;  b2 -= lr * db2
    if step in (0, 1000, 5000):
        print(step, round(loss, 4))

print(P.round(3).ravel())

0 0.6712
1000 0.0051
5000 0.0009
[0.    0.999 0.999 0.002]


In [2]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

X, y = load_digits(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)

mlp = make_pipeline(StandardScaler(),
                    MLPClassifier(hidden_layer_sizes=(64,), max_iter=500, random_state=0))
mlp.fit(X_train, y_train)
print(round(mlp.score(X_train, y_train), 3), round(mlp.score(X_test, y_test), 3))

1.0 0.973


## 14. Going deeper: making deep networks trainable

In [3]:
import numpy as np

# Push a batch through 20 layers of width 256 and watch the activations' spread
rng = np.random.default_rng(0)
x0 = rng.normal(size=(500, 256))
relu = lambda z: np.maximum(z, 0)
setups = {"tanh, weights std 1":           (np.tanh, 1.0),
          "tanh, weights std 0.01":        (np.tanh, 0.01),
          "tanh, Xavier std 1/sqrt(n)":    (np.tanh, 1 / np.sqrt(256)),
          "ReLU, Xavier std 1/sqrt(n)":    (relu, 1 / np.sqrt(256)),
          "ReLU, He std sqrt(2/n)":        (relu, np.sqrt(2 / 256))}
for name, (act, std) in setups.items():
    h = x0; spreads = []
    for layer in range(20):
        h = act(h @ rng.normal(0, std, size=(256, 256)))
        spreads.append(h.std())
    print(f"{name:27}: activation std after layers 1, 10, 20: {spreads[0]:.3f}, {spreads[9]:.3g}, {spreads[19]:.3g}")

# With weights of std 1, tanh is saturated: almost every unit sits near −1 or +1, where its slope is ~0
h = x0
for _ in range(20): h = np.tanh(h @ rng.normal(0, 1.0, size=(256, 256)))
print("tanh with std 1: share of units with |activation| > 0.99:", round(np.mean(np.abs(h) > 0.99), 3))

tanh, weights std 1        : activation std after layers 1, 10, 20: 0.975, 0.974, 0.974
tanh, weights std 0.01     : activation std after layers 1, 10, 20: 0.157, 1.06e-08, 1.12e-16
tanh, Xavier std 1/sqrt(n) : activation std after layers 1, 10, 20: 0.627, 0.221, 0.163


ReLU, Xavier std 1/sqrt(n) : activation std after layers 1, 10, 20: 0.581, 0.0212, 0.000893


ReLU, He std sqrt(2/n)     : activation std after layers 1, 10, 20: 0.829, 0.665, 0.674
tanh with std 1: share of units with |activation| > 0.99: 0.866


In [4]:
# Backpropagate through 20 sigmoid layers vs 20 ReLU layers: how big is the gradient at the first layer?
def grad_norms(act, dact, std):
    h, cache, Ws = x0[:100], [], []
    for _ in range(20):
        W = rng.normal(0, std, size=(256, 256)); z = h @ W
        cache.append(z); Ws.append(W); h = act(z)
    g = np.ones_like(h)                                  # pretend dLoss/dOutput is all ones
    norms = []
    for z, W in zip(reversed(cache), reversed(Ws)):
        g = (g * dact(z)) @ W.T                          # chain rule, one layer back
        norms.append(np.linalg.norm(g))
    return norms[0], norms[-1]

sig = lambda z: 1 / (1 + np.exp(-z))
for name, act, dact, std in [("sigmoid", sig, lambda z: sig(z) * (1 - sig(z)), 1 / np.sqrt(256)),
                             ("ReLU", relu, lambda z: (z > 0).astype(float), np.sqrt(2 / 256))]:
    last, first = grad_norms(act, dact, std)
    print(f"{name:7}: gradient size at the last layer {last:.3g}, at the first layer {first:.3g}")

sigmoid: gradient size at the last layer 40.4, at the first layer 3.5e-11
ReLU   : gradient size at the last layer 186, at the first layer 658


In [5]:
# Inverted dropout: drop units with probability p during training and scale the rest by 1/(1−p)
p = 0.5
h = np.ones(100000)
mask = rng.random(h.shape) > p
train_out = h * mask / (1 - p)
print("share of units kept:", round(mask.mean(), 3), "  average activation, training:", round(train_out.mean(), 3), "  at prediction time (no dropout):", h.mean())

# Batch normalisation: standardise each feature over the batch, then rescale with learned gamma and beta
z = rng.normal(5, 3, size=(64, 4))
gamma, beta = np.array([1.0, 2.0, 1.0, 0.5]), np.array([0.0, 1.0, -1.0, 0.0])
bn = gamma * (z - z.mean(0)) / np.sqrt(z.var(0) + 1e-5) + beta
print("after batch norm, per-feature mean:", (bn.mean(0).round(3) + 0.0), " std:", bn.std(0).round(3))

share of units kept: 0.499   average activation, training: 0.997   at prediction time (no dropout): 1.0
after batch norm, per-feature mean: [ 0.  1. -1.  0.]  std: [1.  2.  1.  0.5]


## Running example: Chennai Homes

The same 12-house dataset used in every lesson (`data/houses.csv` in the course folder).

In [6]:
import pandas as pd
import numpy as np
df = pd.read_csv("../data/houses.csv")
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score, LeaveOneOut
cols = ["size_m2", "dist_km", "age_yrs"]
mlp = make_pipeline(StandardScaler(), MLPClassifier(hidden_layer_sizes=(4,), max_iter=5000, random_state=0))
print("training accuracy:", mlp.fit(df[cols], df["sold_fast"]).score(df[cols], df["sold_fast"]))
print("leave-one-out accuracy:", round(cross_val_score(mlp, df[cols], df["sold_fast"], cv=LeaveOneOut()).mean(), 2))

training accuracy: 1.0


leave-one-out accuracy: 1.0


## Coding challenges

Each cell below has **one missing line**, marked `# ← your code here` (or `# ← replace`). Write the line, run the cell, and the check at the bottom tells you whether you got it right. Stuck? A wrong answer prints a hint, and the solutions are at the very end of this notebook.

In [7]:
# Checker used by the challenges below — run this cell first.
import numpy as np
def check(got, want, hint='', atol=1e-6):
    try:
        ok = got is not None and np.shape(got) == np.shape(want) and np.allclose(np.asarray(got, dtype=float), np.asarray(want, dtype=float), rtol=1e-4, atol=atol)
    except Exception:
        ok = False
    print('✅ Correct!' if ok else '❌ Not yet. Hint: ' + hint)


### Challenge 1: ReLU

Apply ReLU to every value.

In [8]:
import numpy as np
z = np.array([-2.0, 0.0, 3.5])
h = None  # ← your code here
check(h, [0.0, 0.0, 3.5], hint='np.maximum(0, z) keeps positives and turns negatives into 0.')

❌ Not yet. Hint: np.maximum(0, z) keeps positives and turns negatives into 0.


### Challenge 2: A forward pass

Finish the 2-2-1 network from the lesson: compute the output probability p.

In [9]:
import numpy as np
sigmoid = lambda z: 1 / (1 + np.exp(-z))
x = np.array([1.0, 0.0])
W = np.array([[0.5, 0.3], [-0.5, 0.8]])
v = np.array([1.0, -1.0])
h = sigmoid(W @ x)
p = None  # ← your code here
check(p, 0.5609254179424822, hint='The output neuron: sigmoid of v · h. The lesson gets 0.5609.')

❌ Not yet. Hint: The output neuron: sigmoid of v · h. The lesson gets 0.5609.


## Your turn

Try the **practice exercises** at the end of the lesson page. Many of them ask you to change one of the cells above and re-run it.

## Challenge solutions

*Spoilers — try the challenges first!*

**1. ReLU** — the missing line:

```python
h = np.maximum(0, z)
```

**2. A forward pass** — the missing line:

```python
p = sigmoid(v @ h)
```